
# FD004 — BASE vs A vs B (Sensors + Cycle) — CAP=125 — Özcan (2025) Metrikleri

Bu notebook, **FD004** için elindeki **3 farklı preprocessed veri setini** (**BASE**, **A**, **B**) **ayrı ayrı** çalıştırır ve her biri için:

- **Tekil modeller:** LightGBM, CatBoost, GradientBoostingRegressor (GBR) *(opsiyonel: XGBoost)*  
- **Ensemble varyantları (Özcan 2025 yaklaşımı):**
  - **Mean ensemble** (LGBM+Cat)  
  - **Mean ensemble** (LGBM+Cat+GBR)  
  - **Fixed weighted average** (0.7 LGBM + 0.3 Cat)  
  - **Leakage-safe stacking** (GroupKFold(engine_id) ile OOF → Ridge meta-learner)

**Metrikler (Özcan 2025):** R², **MSE**, **MAE**, **RUL Score (PHM08)**  
+ yardımcı: RMSE, τ=15 için (Precision/Recall/Confusion) “karar destek” tanısı.

Çıktılar her varyant için ayrı klasöre yazılır:
- `metrics_<VAR>.csv`, `metrics_<VAR>.json`
- `predictions_<VAR>_<MODEL>.csv`
- `report_<VAR>.md` (tek sayfa özet)
- `run_metadata_<VAR>.json` (konfig + sütunlar + veri boyutları)

> Not: Bu notebook **engine_id’yi feature olarak kullanmaz** (sadece raporlama/gruplama içindir).  
> **cycle** feature olarak dahil edilir.  
> Hedef değişken **CAP=125**: `y = min(RUL, 125)`.

Tarih: 2026-01-28


In [ ]:
# Eğer Colab/temiz ortamdaysan kurulum gerekebilir.
# Local ortamında zaten kuruluysa bu hücreyi geçebilirsin.

%pip -q install lightgbm catboost xgboost scikit-learn pandas numpy


In [ ]:
import os, json, math, warnings, platform, time
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge

import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

warnings.filterwarnings("ignore")

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("lightgbm:", lgb.__version__)
print("catboost:", CatBoostRegressor.__module__)
print("xgboost available:", HAS_XGB)


In [ ]:
# ==============================
# 1) KONFİG — Dosya Yolları
# ==============================
# Bu notebook'ı Windows'ta çalıştıracaksan, aşağıdaki path'leri kendi klasörlerine göre düzenle.
# BASE (global_zscore) dosyalarını sen zaten yükledin; burada örnek olarak relative/absolute verebilirsin.

CAP = 125
TAU = 15  # karar-destek tetik eşiği (Özcan 2025'te örnek olarak 15)

# Çıktı klasörü (her varyant altına klasör açılacak)
OUT_ROOT = Path("fd004_runs_out")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# --- Varyantlar ---
# BASE = global_zscore
# A = zscore_regime_onehot_k6
# B = regimeaware_zscore_k6_onehot

DATASETS = {
    "BASE": {
        "train_csv": "train_FD004_full_global_zscore_nodrop.csv",
        "test_csv": "test_FD004_full_global_zscore_nodrop.csv",
        "scaler_json": "fd004_full_global_zscore_nodrop_scaler.json",  # opsiyonel (meta için)
    },
    "A": {
        # TODO: kendi path'lerinle değiştir
        "train_csv": r"C:\Havelsan\FD004\Veri\FD004_global_zscore_regime_onehot_k6\train_FD004_global_zscore_regime_onehot_k6.csv",
        "test_csv":  r"C:\Havelsan\FD004\Veri\FD004_global_zscore_regime_onehot_k6\test_FD004_global_zscore_regime_onehot_k6.csv",
        "scaler_json": r"C:\Havelsan\FD004\Veri\FD004_global_zscore_regime_onehot_k6\fd004_regime_k6_model.json",  # varsa
    },
    "B": {
        # TODO: kendi path'lerinle değiştir
        "train_csv": r"C:\Havelsan\FD004\Veri\FD004_regimeaware_zscore_k6_onehot\train_FD004_regimeaware_sensor_zscore_k6_onehot.csv",
        "test_csv":  r"C:\Havelsan\FD004\Veri\FD004_regimeaware_zscore_k6_onehot\test_FD004_regimeaware_sensor_zscore_k6_onehot.csv",
        "scaler_json": r"C:\Havelsan\FD004\Veri\FD004_regimeaware_zscore_k6_onehot\fd004_regimeaware_sensor_scaler_k6.json",  # varsa
    },
}

# Büyük veride hız için (istersen True yap)
USE_FLOAT32 = True

# XGBoost'u da denemek ister misin?
RUN_XGBOOST = True


In [ ]:
# ==============================
# 2) Yardımcı Fonksiyonlar
# ==============================

def read_optional_json(path: str):
    try:
        p = Path(path)
        if p.exists():
            return json.loads(p.read_text(encoding="utf-8"))
    except Exception as e:
        return {"_error": str(e)}
    return None

def infer_label_column(df: pd.DataFrame) -> str:
    # Bizim FD004 BASE dosyalarında hedef 'RUL'.
    # A/B dosyalarında isim farklıysa burada genişletilebilir.
    candidates = ["RUL", "rul", "RUL_raw", "RUL_true_raw", "y", "label"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"Label column not found. Available columns: {df.columns.tolist()[:50]} ...")

def build_xy(df: pd.DataFrame, cap: int = 125):
    # engine_id raporlama için; cycle feature olarak dahil.
    label_col = infer_label_column(df)
    engine_id = df["engine_id"].astype(int).values if "engine_id" in df.columns else None
    cycle = df["cycle"].astype(int).values if "cycle" in df.columns else None

    y_raw = df[label_col].astype(float).values
    y = np.minimum(y_raw, cap)

    drop_cols = set([label_col])
    # engine_id feature değil
    if "engine_id" in df.columns:
        drop_cols.add("engine_id")

    # Feature set: geri kalan her şey (cycle dahil kalır)
    X = df.drop(columns=list(drop_cols), errors="ignore")

    # güvenlik: cycle yoksa hata
    if "cycle" not in X.columns:
        raise ValueError("cycle column missing from features; cycle feature is mandatory for this experiment.")

    return X, y, y_raw, engine_id, cycle, label_col

def rul_score_phm08(y_true: np.ndarray, y_pred: np.ndarray, a1: float = 10.0, a2: float = 13.0) -> float:
    # Özcan 2025 Eq. (4): d_i = pred - true
    d = y_pred - y_true
    # early (pred < true) => d < 0
    s = np.where(d < 0, np.exp(-d / a1), np.exp(d / a2))
    return float(np.sum(s))

def tau_diagnostics(y_true: np.ndarray, y_pred: np.ndarray, tau: float = 15.0):
    # bakım yakın = y <= tau
    true_alarm = (y_true <= tau)
    pred_alarm = (y_pred <= tau)

    tp = int(np.sum(true_alarm & pred_alarm))
    fp = int(np.sum(~true_alarm & pred_alarm))
    fn = int(np.sum(true_alarm & ~pred_alarm))
    tn = int(np.sum(~true_alarm & ~pred_alarm))

    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan

    return {
        "tau": tau,
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision": float(precision) if np.isfinite(precision) else None,
        "recall": float(recall) if np.isfinite(recall) else None,
    }

def eval_metrics(y_true: np.ndarray, y_pred: np.ndarray):
    mse = mean_squared_error(y_true, y_pred)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    score = rul_score_phm08(y_true, y_pred)
    return {
        "R2": float(r2),
        "MSE": float(mse),
        "RMSE": float(rmse),
        "MAE": float(mae),
        "RUL_Score_PHM08": float(score),
    }

def safe_read_csv(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"CSV not found: {path}")
    df = pd.read_csv(p)
    if USE_FLOAT32:
        # float sütunları float32'ye çekerek RAM ve hız optimize et
        for c in df.columns:
            if c in ["engine_id", "cycle"]:
                continue
            if pd.api.types.is_float_dtype(df[c]) or pd.api.types.is_integer_dtype(df[c]):
                df[c] = pd.to_numeric(df[c], errors="ignore", downcast="float")
    return df

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p


In [ ]:
# ==============================
# 3) Modeller (Özcan 2025 Table 4 uyumlu parametreler)
# ==============================

def make_models(random_state: int = 42):
    models = {}

    models["LightGBM"] = lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.1,
        max_depth=-1,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=random_state,
        n_jobs=-1,
    )

    models["CatBoost"] = CatBoostRegressor(
        iterations=500,
        learning_rate=0.1,
        depth=6,
        random_seed=random_state,
        verbose=0,
        loss_function="RMSE",
    )

    models["GBR"] = GradientBoostingRegressor(
        n_estimators=300,
        max_depth=7,
        learning_rate=0.1,
        subsample=0.8,
        random_state=random_state,
    )

    if RUN_XGBOOST and HAS_XGB:
        models["XGBoost"] = xgb.XGBRegressor(
            objective="reg:squarederror",
            learning_rate=0.2,
            max_depth=4,
            n_estimators=200,
            subsample=1.0,
            colsample_bytree=0.8,
            random_state=random_state,
            n_jobs=-1,
        )

    return models


In [ ]:
# ==============================
# 4) Tekil Model Eğit + Değerlendir
# ==============================

def train_and_predict(models, X_train, y_train, X_test):
    preds = {}
    fitted = {}
    for name, model in models.items():
        t0 = time.time()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        preds[name] = y_pred
        fitted[name] = model
        print(f"[{name}] fit+predict done in {time.time()-t0:.1f}s")
    return fitted, preds

def save_predictions_csv(out_dir: Path, variant: str, model_name: str, engine_id, cycle, y_true_raw, y_true_cap, y_pred):
    dfp = pd.DataFrame({
        "engine_id": engine_id,
        "cycle": cycle,
        "RUL_true_raw": y_true_raw,
        "RUL_true_capped": y_true_cap,
        "RUL_pred": y_pred
    })
    out_path = out_dir / f"predictions_{variant}_{model_name}.csv"
    dfp.to_csv(out_path, index=False)
    return out_path


In [ ]:
# ==============================
# 5) Ensemble Fonksiyonları
# ==============================

def ensemble_mean(pred_list):
    return np.mean(np.vstack(pred_list), axis=0)

def ensemble_weighted(p1, p2, w1=0.7, w2=0.3):
    return w1*p1 + w2*p2

def stacking_oof_ridge(base_pred_dict_train_oof, y_train, groups, alpha=1.0, random_state=42):
    """Leakage-safe stacking:
    - base_pred_dict_train_oof: {model_name: oof_pred_vector}
    - meta learner: Ridge(alpha=1)
    """
    base_names = list(base_pred_dict_train_oof.keys())
    P = np.vstack([base_pred_dict_train_oof[n] for n in base_names]).T
    meta = Ridge(alpha=alpha, random_state=random_state)
    meta.fit(P, y_train)
    return meta, base_names

def compute_oof_predictions(models, X, y, groups, n_splits=5, random_state=42):
    gkf = GroupKFold(n_splits=n_splits)
    oof = {name: np.zeros(len(y), dtype=float) for name in models.keys()}

    for fold, (tr, va) in enumerate(gkf.split(X, y, groups=groups), 1):
        X_tr, X_va = X.iloc[tr], X.iloc[va]
        y_tr, y_va = y[tr], y[va]

        for name, model in models.items():
            m = model
            m.fit(X_tr, y_tr)
            oof[name][va] = m.predict(X_va)
        print(f"OOF fold {fold}/{n_splits} done")

    return oof


In [ ]:
# ==============================
# 6) Varyantları Koştur (BASE, A, B) + Metrikleri Dosyala
# ==============================

all_summary = []

for variant, cfg in DATASETS.items():
    print("\n" + "="*90)
    print(f"RUN VARIANT: {variant}")
    print("="*90)

    out_dir = ensure_dir(OUT_ROOT / variant)
    train_csv = cfg.get("train_csv")
    test_csv = cfg.get("test_csv")

    # Veri yoksa skip (özellikle A/B'yi path düzeltmeden çalıştırırsan)
    try:
        df_train = safe_read_csv(train_csv)
        df_test = safe_read_csv(test_csv)
    except FileNotFoundError as e:
        print("SKIP:", e)
        continue

    # Build X/y (CAP'li)
    X_train, y_train, y_train_raw, eng_train, cyc_train, label_col = build_xy(df_train, cap=CAP)
    X_test,  y_test,  y_test_raw,  eng_test,  cyc_test,  _        = build_xy(df_test,  cap=CAP)

    # Metadata dump
    meta = {
        "variant": variant,
        "cap": CAP,
        "tau": TAU,
        "train_csv": str(train_csv),
        "test_csv": str(test_csv),
        "scaler_json": str(cfg.get("scaler_json")),
        "label_col": label_col,
        "n_train": int(len(y_train)),
        "n_test": int(len(y_test)),
        "n_features": int(X_train.shape[1]),
        "feature_cols": X_train.columns.tolist(),
        "note": "engine_id feature olarak kullanılmadı; sadece GroupKFold/raporlama için.",
    }
    scaler_meta = read_optional_json(cfg.get("scaler_json", ""))
    if scaler_meta is not None:
        meta["scaler_meta"] = scaler_meta
    (out_dir / f"run_metadata_{variant}.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

    # Modeller
    models = make_models(random_state=42)

    # Tekil eğitim/test
    fitted, preds_test = train_and_predict(models, X_train, y_train, X_test)

    # Tekil metrikler + predictions export
    rows = []
    for name, y_pred in preds_test.items():
        m = eval_metrics(y_test, y_pred)
        td = tau_diagnostics(y_test, y_pred, tau=TAU)
        row = {"variant": variant, "model": name, **m, **{f"tau15_{k}": v for k,v in td.items() if k!='tau'}}
        rows.append(row)

        save_predictions_csv(out_dir, variant, name, eng_test, cyc_test, y_test_raw, y_test, y_pred)

    # Ensemble'lar (Özcan stili)
    # 2-model mean/weighted: LGBM + CatBoost
    if "LightGBM" in preds_test and "CatBoost" in preds_test:
        y_mean_2 = ensemble_mean([preds_test["LightGBM"], preds_test["CatBoost"]])
        y_w70_30 = ensemble_weighted(preds_test["LightGBM"], preds_test["CatBoost"], 0.7, 0.3)

        for en_name, y_pred in [("ENS_mean_LGBM_Cat", y_mean_2), ("ENS_w70_LGBM_w30_Cat", y_w70_30)]:
            m = eval_metrics(y_test, y_pred)
            td = tau_diagnostics(y_test, y_pred, tau=TAU)
            rows.append({"variant": variant, "model": en_name, **m, **{f"tau15_{k}": v for k,v in td.items() if k!='tau'}})
            save_predictions_csv(out_dir, variant, en_name, eng_test, cyc_test, y_test_raw, y_test, y_pred)

    # 3-model mean: LGBM + Cat + GBR
    if all(k in preds_test for k in ["LightGBM", "CatBoost", "GBR"]):
        y_mean_3 = ensemble_mean([preds_test["LightGBM"], preds_test["CatBoost"], preds_test["GBR"]])
        m = eval_metrics(y_test, y_mean_3)
        td = tau_diagnostics(y_test, y_mean_3, tau=TAU)
        rows.append({"variant": variant, "model": "ENS_mean_LGBM_Cat_GBR", **m, **{f"tau15_{k}": v for k,v in td.items() if k!='tau'}})
        save_predictions_csv(out_dir, variant, "ENS_mean_LGBM_Cat_GBR", eng_test, cyc_test, y_test_raw, y_test, y_mean_3)

    # Leakage-safe stacking (OOF + Ridge) — base: LGBM, Cat, GBR (varsa)
    stack_base_names = [n for n in ["LightGBM", "CatBoost", "GBR"] if n in models]
    if len(stack_base_names) >= 2:
        print("\n[Stacking] OOF predictions with GroupKFold(engine_id) ...")
        base_models_for_oof = {n: models[n] for n in stack_base_names}
        # OOF (train)
        oof = compute_oof_predictions(base_models_for_oof, X_train, y_train, groups=eng_train, n_splits=5)
        # meta train
        meta_model, base_order = stacking_oof_ridge(oof, y_train, groups=eng_train, alpha=1.0)

        # Fit base models on full train
        for n in base_order:
            base_models_for_oof[n].fit(X_train, y_train)
        # Test stacking features
        P_test = np.vstack([base_models_for_oof[n].predict(X_test) for n in base_order]).T
        y_stack = meta_model.predict(P_test)

        m = eval_metrics(y_test, y_stack)
        td = tau_diagnostics(y_test, y_stack, tau=TAU)
        rows.append({"variant": variant, "model": f"STACK_Ridge_on_{'+'.join(base_order)}", **m, **{f"tau15_{k}": v for k,v in td.items() if k!='tau'}})
        save_predictions_csv(out_dir, variant, f"STACK_Ridge_on_{'+'.join(base_order)}", eng_test, cyc_test, y_test_raw, y_test, y_stack)

    # Metrikleri dosyala
    dfm = pd.DataFrame(rows).sort_values(by=["RMSE", "RUL_Score_PHM08"], ascending=[True, True])
    dfm_path_csv = out_dir / f"metrics_{variant}.csv"
    dfm_path_json = out_dir / f"metrics_{variant}.json"
    dfm.to_csv(dfm_path_csv, index=False)
    dfm.to_json(dfm_path_json, orient="records", force_ascii=False, indent=2)

    # Tek sayfalık MD rapor
    best = dfm.iloc[0].to_dict() if len(dfm) else {}
    md_lines = []
    md_lines.append(f"# FD004 — {variant} — Metrics Summary (CAP={CAP})")
    md_lines.append("\n## Best Model (by RMSE then RUL Score)\n")
    for k in ["model","RMSE","MSE","MAE","R2","RUL_Score_PHM08","tau15_precision","tau15_recall","tau15_TP","tau15_FP","tau15_FN","tau15_TN"]:
        if k in best:
            md_lines.append(f"- **{k}**: {best[k]}")
    md_lines.append("\n## Full Metrics Table (top 15)\n")
    md_lines.append(dfm.head(15).to_markdown(index=False))
    (out_dir / f"report_{variant}.md").write_text("\n".join(md_lines), encoding="utf-8")

    print("\nTop-5 by RMSE:")
    display(dfm.head(5))

    # Summary row for global compare
    best_row = dfm.iloc[0].copy()
    best_row["variant"] = variant
    all_summary.append(best_row)

# Global compare
if all_summary:
    df_all = pd.DataFrame(all_summary).sort_values(by=["RMSE", "RUL_Score_PHM08"], ascending=[True, True])
    df_all.to_csv(OUT_ROOT / "BEST_PER_VARIANT.csv", index=False)
    df_all.to_json(OUT_ROOT / "BEST_PER_VARIANT.json", orient="records", force_ascii=False, indent=2)
    print("\n=== BEST PER VARIANT ===")
    display(df_all)
else:
    print("\nNo variants were run. Check DATASETS paths.")


## Notlar / Beklenen Davranış

- **A** ve **B** varyantlarını burada *placeholder* path ile bıraktım; kendi bilgisayarındaki klasörlere göre güncelle.
- Varyantların sütun sayısı farklı olabilir (one-hot vs regime-aware). Bu notebook **etiket sütunu hariç** tüm sütunları feature kabul eder; `engine_id` feature değildir.
- **Stacking leakage** için meta-learner **OOF tahminler** üzerinden eğitilir (GroupKFold(engine_id)). Bu, “test bilgisi”nin meta seviyeye sızmasını engeller.
- Eğer istersen Ozcan’ın raporladığı **FD004 referans değerleri** ile karşılaştırma satırını da rapora ekleyebiliriz (paper Table 8 / FD004 satırları).

✅ Çalıştırdıktan sonra bana şu dosyaları atarsan, aynı FD003’te yaptığımız gibi Word/PDF raporu da üretebilirim:
- `fd004_runs_out/BASE/metrics_BASE.csv`
- `fd004_runs_out/A/metrics_A.csv`
- `fd004_runs_out/B/metrics_B.csv`
- En iyi modelin `predictions_*.csv` dosyası
